In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from collections import defaultdict



url = "https://raw.githubusercontent.com/komaksym/UFC-DataLab/main/data/merged_stats_n_scorecards/merged_stats_n_scorecards.csv"
raw = pd.read_csv(url)


In [ ]:
df = raw.copy()
df['date'] = pd.to_datetime(df['event_date'], format='%d/%m/%Y', errors='coerce')

#Makes fighter names uppercase
for team in ['red', 'blue']:
    df[team] = df[team + '_fighter_name'].astype('string').str.strip().str.upper()

#Removes fights with missing dates or fighter names
df = df.dropna(subset=['date','red','blue'])

#Sorts the fighters names and combines them
def createPair(row):
    fighters = [row['red'], row['blue']]
    fighters.sort()
    return '|'.join(fighters)

#Creates a pair column for each fight
df['pair'] = df.apply(createPair, axis=1)

#Removes duplicate matchups that happened on the same date
df = df.drop_duplicates(['date','pair'])
df = df[df.fight_outcome.isin(['red_win','blue_win'])].sort_values('date')

print('fights:', len(df))
print('Fights from:', df.date.min().date(), 'to', df.date.max().date())


fights: 8580
Fights from: 1994-03-11 to 2026-06-27


In [ ]:
#Stores each fighters previous results and stats
fightHistory = defaultdict(list)
fightStats = ['sig_str', 'TD', 'KD', 'sub_att']

#Turns stats like 50 of 100 into 50
def number(value):
    value = str(value).split(' of ')[0]
    return pd.to_numeric(value, errors='coerce')

#Returns the fighters previous fight count and average stats
def priorFights(name):
    history = fightHistory[name]

    if len(history) == 0:
        return np.array([0, np.nan, np.nan, np.nan, np.nan, np.nan])

    averages = pd.DataFrame(history).mean().tolist()
    return np.array([len(history)] + averages)

rows = []

#Processes fights from oldest to newest
for date, fights in df.groupby('date', sort=True):
    updates = []

    for _, fight in fights.iterrows():
        red = fight['red']
        blue = fight['blue']

        if fight['fight_outcome'] == 'red_win':
            red_wins = 1
        else:
            red_wins = 0

        #Compares the fighters stats from earlier fights
        statDiff = priorFights(red) - priorFights(blue)
        newRow = [date, red, blue, red_wins]

        for difference in statDiff:
            newRow.append(difference)

        rows.append(newRow)

        #Collects both fighters results and stats from this fight
        redStats = [red_wins]
        blueStats = [1 - red_wins]

        for stat in fightStats:
            redStats.append(number(fight['red_fighter_' + stat]))
            blueStats.append(number(fight['blue_fighter_' + stat]))

        updates.append((red, redStats))
        updates.append((blue, blueStats))

    #Adds results to history after all fights on this date are processed
    for name, stats in updates:
        fightHistory[name].append(stats)

#Creates the table used to train and test the model
features = [
    'prior_bouts_diff',
    'prior_win_rate_diff',
    'prior_sig_landed_diff',
    'prior_td_landed_diff',
    'prior_knockdowns_diff',
    'prior_sub_attempts_diff'
]

columns = ['date', 'fighter_A', 'fighter_B', 'A_wins'] + features
data = pd.DataFrame(rows, columns=columns)



In [ ]:
#Uses latest 20% for testing and the rest for training
split = int(len(data) * 0.8)
cutoff = data.iloc[split]['date']

train = data[data['date'] < cutoff]
test = data[data['date'] >= cutoff]

#Separates the stats from the fight results
XTrain = train[features]
yTrain = train['A_wins']

XTest = test[features]
yTest = test['A_wins']

model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

#Trains the model using earlier fights
model.fit(XTrain, yTrain)

#Predicts the winners of the test fights
pred = model.predict(XTest)

print('Training fights:', len(train))
print('Test fights:', len(test))


Training fights: 6860
Test fights: 1720


In [ ]:
baseline = np.full(len(test), yTrain.mode().iloc[0])

#Compares the models accuracy to always predicting the most common training result
print('Baseline accuracy:', round(accuracy_score(yTest, baseline), 4))
print('Logistic regression accuracy:', round(accuracy_score(yTest, pred), 4))

#Shows correct and incorrect predictions for each winner
print('Confusion matrix (0 = B wins, 1 = A wins):')
print(confusion_matrix(yTest, pred))


Baseline accuracy: 0.5593
Logistic regression accuracy: 0.5674
Confusion matrix (0 = B wins, 1 = A wins):
[[ 41 717]
 [ 27 935]]


In [ ]:
# @markdown Run this cell to download this notebook as a webpage, `_NOTEBOOK.html`.

import google, json, nbformat

# Get the current notebook and write it to _NOTEBOOK.ipynb
raw_notebook = google.colab._message.blocking_request("get_ipynb",
                                                      timeout_sec=30)["ipynb"]
with open("_NOTEBOOK.ipynb", "w", encoding="utf-8") as ipynb_file:
  ipynb_file.write(json.dumps(raw_notebook))

# Use nbconvert to convert .ipynb to .html.
!jupyter nbconvert --to html --log-level WARN _NOTEBOOK.ipynb

# Download the .html file.
google.colab.files.download("_NOTEBOOK.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>